# Algorithmic Trading and Quantitative Strategies
## Part 12: Robustness Testing and Statistical Validation
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. **Dimensions of Robustness**
   - 1.1 Parameter Sensitivity
   - 1.2 Regime Sensitivity
   - 1.3 Skipping Top Trades
   - 1.4 Fee Sensitivity
   - 1.5 Sample Period Sensitivity
   - 1.6 Parameter Sensitivity Analysis (code)
   - 1.7 Fee Sensitivity (code)
   - 1.8 Sub-Period / Regime Sensitivity (code)
2. **Statistical Testing for Trading Strategy Performance**
   - 2.1 Population vs. Sample
   - 2.2 Sampling Distribution
   - 2.3 Inference With Only One Sample
   - 2.4 Parametric Approach for Strategy Returns
   - 2.5 Confidence Intervals for the Sharpe Ratio
3. **Bootstrap Approach**
   - 3.1 Simple Bootstrap
   - 3.2 Block Bootstrap (Simple-Block & Moving-Block)
4. **Testing Skill vs. Luck (Monte Carlo)**
   - 4.1 Motivating Example
   - 4.2 Sequential Testing
   - 4.3 Hypothesis Testing Framework
   - 4.4 Method 1: Random {-1, 0, +1}
   - 4.5 Method 2: Random Long-Only {0, +1}
   - 4.6 Method 3: Original Positions x Reshuffled Returns
   - 4.7 Comparison of the Three Methods
5. **Application to a Backtrader Strategy**
6. **Exercises**

## 1. Dimensions of Robustness

A robust strategy delivers consistent performance across different conditions. We test robustness along several dimensions:

### 1.1 Parameter Sensitivity

A good strategy should perform well across a range of parameters — not just at one optimal point. If performance drops dramatically with small parameter changes, the strategy is likely overfit.

### 1.2 Regime Sensitivity

Test across different market regimes:
- Bull vs. bear markets
- High vs. low volatility periods
- Different economic cycles

### 1.3 Skipping Top Trades

Remove the best N trades and re-evaluate. If the strategy's performance depends entirely on a few lucky trades, it may not be reliable.

### 1.4 Fee Sensitivity

Test with varying levels of transaction costs (0 bps, 5 bps, 10 bps, 20 bps). Some strategies are only profitable with unrealistically low costs.

### 1.5 Sample Period Sensitivity

Test on different time periods (sub-samples). Performance should be relatively consistent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import backtrader as bt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)

# Download data
spy_data = yf.download('SPY', start='2010-01-01', end='2024-12-31')
spy_data.columns = spy_data.columns.droplevel('Ticker')
spy_returns = spy_data['Close'].pct_change().dropna()
print(f"SPY: {len(spy_returns)} daily returns")

### 1.6 Parameter Sensitivity Analysis

In [ ]:
class SMAStrategy(bt.Strategy):
    params = dict(fast=20, slow=50)
    
    def __init__(self):
        fast_ma = bt.ind.SMA(self.data.close, period=self.p.fast)
        slow_ma = bt.ind.SMA(self.data.close, period=self.p.slow)
        self.crossover = bt.ind.CrossOver(fast_ma, slow_ma)
        self.order = None
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

def run_bt_strategy(data_df, fast, slow, cash=100000, commission=0.001):
    """Run SMA strategy and return key metrics."""
    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=data_df))
    cerebro.addstrategy(SMAStrategy, fast=fast, slow=slow)
    cerebro.broker.setcash(cash)
    cerebro.broker.setcommission(commission=commission)
    cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='dd')
    
    results = cerebro.run()
    strat = results[0]
    sr = strat.analyzers.sharpe.get_analysis().get('sharperatio', 0) or 0
    ret = strat.analyzers.returns.get_analysis().get('rtot', 0)
    dd = strat.analyzers.dd.get_analysis().max.drawdown
    
    return {'return': ret, 'sharpe': sr, 'max_dd': dd, 'final_value': cerebro.broker.getvalue()}

# Parameter sensitivity
print("Running parameter sensitivity analysis...")
sensitivity = []
for fast in range(10, 55, 5):
    for slow in range(60, 210, 10):
        result = run_bt_strategy(spy_data, fast, slow)
        result['fast'] = fast
        result['slow'] = slow
        sensitivity.append(result)

sens_df = pd.DataFrame(sensitivity)

# Heatmap of Sharpe ratios
pivot = sens_df.pivot_table(values='sharpe', index='fast', columns='slow')

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', origin='lower')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns.astype(int), rotation=45, fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index.astype(int), fontsize=8)
ax.set_xlabel('Slow Period')
ax.set_ylabel('Fast Period')
ax.set_title('Parameter Sensitivity: Sharpe Ratio')
fig.colorbar(im)
plt.tight_layout()
plt.show()

# Percentage of parameter combinations that are profitable
profitable = (sens_df['return'] > 0).mean()
print(f"\n{profitable:.1%} of parameter combinations are profitable")
print(f"Median Sharpe: {sens_df['sharpe'].median():.3f}")
print(f"Sharpe range: [{sens_df['sharpe'].min():.3f}, {sens_df['sharpe'].max():.3f}]")

### 1.7 Fee Sensitivity

In [ ]:
# Test with different commission levels
commissions = [0, 0.0005, 0.001, 0.002, 0.005]
fast, slow = 20, 100

print(f"SMA({fast}/{slow}) Strategy - Fee Sensitivity:")
print(f"{'Commission':>12} {'Return':>10} {'Sharpe':>10} {'Max DD':>10}")
print("-" * 45)

for comm in commissions:
    result = run_bt_strategy(spy_data, fast, slow, commission=comm)
    print(f"{comm:>11.2%} {result['return']:>10.2%} {result['sharpe']:>10.3f} "
          f"{result['max_dd']:>9.1f}%")

### 1.8 Sub-Period / Regime Sensitivity

A robust strategy should perform reasonably well across different sub-periods (different market regimes). We split the backtest into non-overlapping sub-periods and re-evaluate.

If performance is concentrated in one regime (e.g., post-2020), the strategy may not generalize.

In [ ]:
# Sub-period sensitivity: split SPY data into 3 equal periods and re-run SMA(20/100)
fast, slow = 20, 100
n_periods = 3
splits = np.array_split(np.arange(len(spy_data)), n_periods)

print(f"SMA({fast}/{slow}) Strategy - Sub-Period Sensitivity (3 sub-samples):")
print(f"{'Period':<25} {'Return':>10} {'Sharpe':>10} {'Max DD':>10}")
print("-" * 58)

# Full-sample reference
full = run_bt_strategy(spy_data, fast, slow)
print(f"{'Full Sample':<25} {full['return']:>10.2%} {full['sharpe']:>10.3f} "
      f"{full['max_dd']:>9.1f}%")

sub_results = []
for i, idx in enumerate(splits):
    sub = spy_data.iloc[idx[0]:idx[-1] + 1]
    res = run_bt_strategy(sub, fast, slow)
    label = f"P{i+1}: {sub.index[0].date()}..{sub.index[-1].date()}"
    sub_results.append((label, res))
    print(f"{label:<25} {res['return']:>10.2%} {res['sharpe']:>10.3f} "
          f"{res['max_dd']:>9.1f}%")

# Plot sub-period Sharpes
fig, ax = plt.subplots(figsize=(9, 3.5))
labels = ['Full'] + [f'P{i+1}' for i in range(n_periods)]
sharpes = [full['sharpe']] + [r[1]['sharpe'] for r in sub_results]
colors = ['steelblue'] + ['darkorange'] * n_periods
ax.bar(labels, sharpes, color=colors)
ax.axhline(0, color='k', linewidth=0.7)
ax.set_title(f'SMA({fast}/{slow}) Sharpe Ratio by Sub-Period')
ax.set_ylabel('Sharpe (annualized, from analyzer)')
plt.tight_layout()
plt.show()

## 2. Statistical Testing for Trading Strategy Performance

### 2.1 Population vs. Sample

Assume that $Y$ represents a collection of random variables $Y_1, Y_2, \ldots$ which are **independent and identically distributed** (i.i.d.). That is, each random variable $Y_i \sim F(\Theta)$ has the same probability distribution (with parameter $\Theta$) and all are mutually independent.

Our aim is to make inferences about the **true (unobserved) population parameter** $\Theta$ from sample data. We rely on a sample statistic $\theta$ that is an *unbiased estimator* of $\Theta$:

$$E[\theta] = \Theta$$

**Example.** If $Y_i \sim N(\mu, \sigma)$, then for a realization $Y_1, \ldots, Y_N$, the sample mean

$$m = \frac{1}{N}\sum_{i=1}^N Y_i$$

is an unbiased estimator of $\mu$:

$$E[m] = \tfrac{1}{N}\sum E[Y_i] = \tfrac{1}{N}\sum \mu = \mu$$

### 2.2 Sampling Distribution

The sample mean $m$ is itself a random variable &mdash; each time we draw a sample, we get a new realization of $m$, possibly with a different value. The distribution of $m$ across many samples is called the **sampling distribution**.

For a Gaussian population $Y_i \sim N(\mu, \sigma)$:

$$E[m] = \mu, \qquad \mathrm{Var}[m] = \frac{\sigma^2}{N}$$

So the sampling distribution of the sample mean is

$$m \;\sim\; N\!\left(\mu, \tfrac{\sigma}{\sqrt{N}}\right)$$

### 2.3 Inference With Only One Sample

In trading, our backtest result is **just one sample** from the (unobservable) population of strategy returns. If we ran the strategy in another period or on another asset, returns would differ &mdash; another realization from the same underlying population.

When we have only one sample, we can use two approaches:

- **Parametric approach** &mdash; assume a parametric form for the population distribution (e.g., Gaussian), then derive the sampling distribution analytically.
- **Bootstrap approach** &mdash; resample from the observed data to *empirically* approximate the sampling distribution. (Section 3.)

### 2.4 Parametric Approach for Strategy Returns

Given a strategy with daily returns $r_1, r_2, \ldots, r_T$:

$$H_0: \mu = 0 \quad \text{(strategy has no skill)}$$
$$H_1: \mu \neq 0 \quad \text{(strategy has skill)}$$

**Test statistic** (assuming returns are i.i.d. Normal):

$$t = \frac{\bar{r}}{s / \sqrt{T}}$$

where $\bar{r}$ is the sample mean, $s$ is the sample standard deviation, and $T$ is the number of observations.

**Confidence interval for the mean:**  $\bar{r} \pm z_{\alpha/2}\, \tfrac{s}{\sqrt{T}}$.

### 2.5 Confidence Intervals for the Sharpe Ratio

The Sharpe ratio $\hat{SR} = \frac{\bar{r}}{s}$ has an approximate (Lo, 2002) standard error:

$$SE(\hat{SR}) \;\approx\; \sqrt{\frac{1 + \tfrac{1}{2}\hat{SR}^2}{T}}$$

A 95% confidence interval is given by $\hat{SR} \pm 1.96 \cdot SE(\hat{SR})$.

In [ ]:
# Demo: construct the sampling distribution of the sample mean
# Population: Y ~ N(mu = 0.10, sigma = 0.25). Sample size N = 100.
np.random.seed(1)
mu, sigma, N, M = 0.10, 0.25, 100, 10_000

sample_means = np.array([np.random.normal(mu, sigma, N).mean() for _ in range(M)])

print(f"Population:  mu = {mu}, sigma = {sigma}")
print(f"Drew M = {M:,} samples of size N = {N}")
print(f"Sample-mean estimate of mu:  {sample_means.mean():.5f}")
print(f"Theoretical SE(m) = sigma/sqrt(N) = {sigma/np.sqrt(N):.5f}")
print(f"Empirical std of sample means: {sample_means.std():.5f}")
print(f"5%-95% empirical CI for m:    [{np.percentile(sample_means, 5):.4f}, "
      f"{np.percentile(sample_means, 95):.4f}]")

# Plot the sampling distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sample_means, bins=80, density=True, alpha=0.7, color='steelblue',
        edgecolor='white', label='Empirical sampling distribution')
xs = np.linspace(sample_means.min(), sample_means.max(), 300)
ax.plot(xs, stats.norm.pdf(xs, loc=mu, scale=sigma/np.sqrt(N)),
        color='red', lw=2, label='Theoretical N(mu, sigma/sqrt(N))')
ax.axvline(mu, color='black', linestyle='--', label=f'True mu = {mu}')
ax.set_title('Sampling Distribution of the Sample Mean')
ax.set_xlabel('Sample mean m')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def parametric_test(returns, name="Strategy"):
    """Perform parametric statistical tests on strategy returns."""
    n = len(returns)
    mean_ret = returns.mean()
    std_ret = returns.std()
    
    # t-test for mean return
    t_stat, p_value = stats.ttest_1samp(returns, 0)
    
    # Annualized Sharpe
    sharpe = mean_ret / std_ret * np.sqrt(252)
    
    # SE of Sharpe ratio
    se_sharpe = np.sqrt((1 + 0.5 * sharpe**2) / n) * np.sqrt(252)
    ci_lower = sharpe - 1.96 * se_sharpe
    ci_upper = sharpe + 1.96 * se_sharpe
    
    print(f"{'='*50}")
    print(f"Parametric Tests: {name}")
    print(f"{'='*50}")
    print(f"Observations:         {n}")
    print(f"Daily Mean Return:    {mean_ret:.6f}")
    print(f"Daily Std:            {std_ret:.6f}")
    print(f"t-statistic:          {t_stat:.4f}")
    print(f"p-value:              {p_value:.4f}")
    print(f"Significant (5%):     {'Yes' if p_value < 0.05 else 'No'}")
    print(f"\nAnnualized Sharpe:    {sharpe:.4f}")
    print(f"SE(Sharpe):           {se_sharpe:.4f}")
    print(f"95% CI for Sharpe:    [{ci_lower:.4f}, {ci_upper:.4f}]")
    
    return {'t_stat': t_stat, 'p_value': p_value, 'sharpe': sharpe, 
            'ci_lower': ci_lower, 'ci_upper': ci_upper}

# Generate strategy returns (using a simple SMA strategy)
# Simulate strategy returns for demonstration
np.random.seed(42)
strategy_returns = pd.Series(
    np.random.normal(0.0003, 0.012, 2500),  # 10 years of daily returns
    index=pd.date_range('2015-01-01', periods=2500, freq='B')
)

results = parametric_test(strategy_returns, "SMA Crossover Strategy")

## 3. Bootstrap Approach

The bootstrap is the empirical alternative to the parametric approach. We **do not** assume a parametric form for the return distribution. Instead, we resample directly from the observed returns to construct the sampling distribution of any statistic of interest.

### 3.1 Simple Bootstrap

**Steps:**

1. Randomly sample (with replacement) from the strategy returns to generate a new return series of the same length:
   $$r^*_{b,1}, r^*_{b,2}, \ldots, r^*_{b,T}$$
2. Compute the statistic of interest (Sharpe, max drawdown, return, ...) on the bootstrap sample:
   $$\hat{\theta}^*_b = g(r^*_{b,1}, \ldots, r^*_{b,T}), \quad b = 1, \ldots, B$$
3. Repeat $B$ times to get an empirical sampling distribution of $\hat{\theta}$.
4. Construct confidence intervals from the empirical distribution.

### 3.2 Block Bootstrap

The simple bootstrap assumes returns are i.i.d. If returns are **serially correlated**, simple resampling destroys the temporal dependence structure. **Block bootstrap** preserves it by resampling **blocks** of consecutive returns:

- **Simple Block Bootstrap.** Split the observed sample into *non-overlapping* blocks of length $b$. Randomly sample (with replacement) among these blocks and concatenate them.
- **Moving Block Bootstrap.** Split the observed sample into $n - b + 1$ *overlapping* blocks of length $b$ (block 1: obs $1..b$; block 2: obs $2..b+1$; ...). Then draw $n/b$ blocks at random with replacement and concatenate them.

The block size $b$ is a key tuning parameter:

- Too small: serial dependence is lost (close to simple bootstrap).
- Too large: too few blocks, loss of variation across resamples.

In [ ]:
def _stat(boot_sample, statistic, ann_factor=252):
    if statistic == 'sharpe':
        s = boot_sample.std()
        return boot_sample.mean() / s * np.sqrt(ann_factor) if s > 0 else 0.0
    if statistic == 'return':
        return boot_sample.mean() * ann_factor
    if statistic == 'max_drawdown':
        cum = np.cumprod(1 + boot_sample)
        return (cum / np.maximum.accumulate(cum) - 1).min()
    raise ValueError(statistic)


def simple_bootstrap(returns, n_bootstrap=10000, statistic='sharpe'):
    """Simple i.i.d. bootstrap (sample individual returns with replacement)."""
    rets = np.asarray(returns)
    n = len(rets)
    return np.array([_stat(np.random.choice(rets, size=n, replace=True), statistic)
                     for _ in range(n_bootstrap)])


def simple_block_bootstrap(returns, n_bootstrap=10000, block_size=20, statistic='sharpe'):
    """
    Simple Block Bootstrap: split into NON-OVERLAPPING blocks, sample blocks with replacement.
    """
    rets = np.asarray(returns)
    n = len(rets)
    n_full_blocks = n // block_size
    blocks = rets[:n_full_blocks * block_size].reshape(n_full_blocks, block_size)
    n_needed = n // block_size + 1
    out = np.zeros(n_bootstrap)
    for b in range(n_bootstrap):
        idx = np.random.randint(0, n_full_blocks, size=n_needed)
        boot_sample = blocks[idx].ravel()[:n]
        out[b] = _stat(boot_sample, statistic)
    return out


def moving_block_bootstrap(returns, n_bootstrap=10000, block_size=20, statistic='sharpe'):
    """
    Moving Block Bootstrap: blocks are OVERLAPPING (n - b + 1 of them).
    Draw n/b blocks at random with replacement and concatenate.
    """
    rets = np.asarray(returns)
    n = len(rets)
    n_needed = n // block_size + 1
    out = np.zeros(n_bootstrap)
    for b in range(n_bootstrap):
        starts = np.random.randint(0, n - block_size + 1, size=n_needed)
        boot_sample = np.concatenate([rets[s:s + block_size] for s in starts])[:n]
        out[b] = _stat(boot_sample, statistic)
    return out


# Run bootstrap on strategy returns
B = 5000
print(f"Running three bootstraps ({B:,} iterations each) ...")
boot_sr_simple = simple_bootstrap(strategy_returns.values, B, 'sharpe')
boot_sr_sblock = simple_block_bootstrap(strategy_returns.values, B, 20, 'sharpe')
boot_sr_mblock = moving_block_bootstrap(strategy_returns.values, B, 20, 'sharpe')

for label, arr in [('Simple', boot_sr_simple),
                   ('Simple-Block (b=20)', boot_sr_sblock),
                   ('Moving-Block (b=20)', boot_sr_mblock)]:
    ci = np.percentile(arr, [2.5, 97.5])
    print(f"\n  {label} bootstrap of Sharpe:")
    print(f"    Mean:   {arr.mean():.4f}")
    print(f"    95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
    print(f"    P(SR > 0): {(arr > 0).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for arr, label, color in [(boot_sr_simple,    'Simple',         'steelblue'),
                          (boot_sr_sblock,    'Simple-Block',   'darkorange'),
                          (boot_sr_mblock,    'Moving-Block',   'seagreen')]:
    axes[0].hist(arr, bins=80, density=True, alpha=0.5, label=label, color=color)
axes[0].axvline(0, color='k', linewidth=1.5, label='SR = 0')
axes[0].set_title('Bootstrap Sampling Distribution: Sharpe Ratio')
axes[0].set_xlabel('Sharpe Ratio')
axes[0].legend()

# Bootstrap distribution for max drawdown (matches the PDF MA-crossover example)
boot_dd = simple_bootstrap(strategy_returns.values, B, 'max_drawdown')
ci_dd = np.percentile(boot_dd, [5, 95])
axes[1].hist(boot_dd, bins=80, density=True, alpha=0.7, color='crimson')
axes[1].axvline(ci_dd[0], color='k', linestyle='--', label=f'5%-95% CI')
axes[1].axvline(ci_dd[1], color='k', linestyle='--')
axes[1].set_title('Bootstrap Sampling Distribution: Max Drawdown')
axes[1].set_xlabel('Max Drawdown')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nMax Drawdown 5%-95% CI: [{ci_dd[0]:.4f}, {ci_dd[1]:.4f}]")
print(f"Point estimate (single sample): "
      f"{_stat(strategy_returns.values, 'max_drawdown'):.4f}")

# Keep these aliases for use in later cells
boot_sharpe = boot_sr_simple
block_bootstrap = moving_block_bootstrap  # back-compat alias

## 4. Testing Skill vs. Luck (Monte Carlo Approach)

A profitable backtest does **not** automatically mean the strategy has *skill*. The performance might simply be the result of *luck*, or favorable market conditions during the backtest period.

### 4.1 Motivating Example

Suppose you design a trend-following strategy and backtest it on the S&P 500 for **2010-2016**. The backtest shows:

- Total return: **70%**
- Daily returns are statistically significantly positive (t-test p < 0.01)

You think you found the way to become rich. Then you realize that during the same period:

- The S&P 500 (buy & hold) returned **100%**

In a strong bull market, **any** random strategy with a long bias may yield superior absolute returns. So a good backtest does NOT automatically imply skill.

The relative ranking we want to see is:

> Any random model  &nbsp;&lt;&nbsp;  Any random model **with a long bias**  &nbsp;&lt;&nbsp;  **Our system** (an intelligent system with a long bias)

### 4.2 Sequential Testing

To differentiate between skill and luck, we test our strategy sequentially against three increasingly demanding null hypotheses:

1. **Method 1** &mdash; Any random strategy (no constraints)
2. **Method 2** &mdash; Any random strategy with the **same constraints** (e.g., long-only)
3. **Method 3** &mdash; Any random strategy with the same constraints **AND same trade characteristics** (e.g., same fraction of long days, same trade durations)

A robust strategy should pass *all three* tests.

### 4.3 Hypothesis Testing Framework

For each of the three methods, we set up:

- **Null hypothesis** $H_0$: Our strategy is no better than the corresponding random strategy
- **Alternative hypothesis** $H_A$: Our strategy is better than the corresponding random strategy

**Steps:**

1. Define the random strategy population (Method 1, 2, or 3 below)
2. Generate $M$ random strategies under $H_0$ and compute the performance statistic (e.g., Sharpe ratio) for each
3. Construct the sampling distribution under $H_0$
4. Compute the **p-value** &mdash; the fraction of random strategies whose Sharpe is at least as high as ours (right-tailed test)
5. If p-value $< \alpha$ (e.g., 0.05), reject $H_0$ and conclude our strategy has skill

#### Common construction: reshuffled underlying returns

In all three methods, we **reshuffle** the daily returns of the **underlying asset**:

$$\tilde{r}_t \;=\; \text{random permutation of } r_t^{\text{underlying}}$$

This destroys any temporal predictability while preserving the marginal distribution of returns. The strategy returns under $H_0$ are then:

$$r_t^{\text{Strategy},\,H_0} \;=\; \text{Pos}_t \cdot \tilde{r}_t$$

The three methods differ only in **how the position series $\text{Pos}_t$ is generated**.

### 4.4 Method 1 &mdash; Random Positions $\in \{-1, 0, +1\}$

$$H_0: \text{Our strategy is no better than ANY random trading strategy}$$

For each day, randomly draw the position from $\{-1, 0, +1\}$:

- $-1$: short
- $\phantom{-}0$: neutral (cash)
- $+1$: long

Combined with reshuffled underlying returns, this is the **most permissive** null hypothesis. If our strategy beats this benchmark, it shows the strategy does *something* &mdash; but it could simply be exploiting a directional bias in the underlying market.

In [ ]:
def mc_method1_random_long_short_neutral(underlying_returns, n_periods,
                                         n_simulations=5000, ann_factor=252):
    """
    Method 1: Random positions from {-1, 0, +1} multiplied by reshuffled returns.

    H0: Our strategy is no better than any random trading strategy.
    """
    rets = np.asarray(underlying_returns)
    sharpes = np.zeros(n_simulations)

    for sim in range(n_simulations):
        positions = np.random.choice([-1, 0, 1], size=n_periods)
        shuffled_rets = np.random.permutation(rets)[:n_periods]
        strat_rets = positions * shuffled_rets
        s = strat_rets.std()
        sharpes[sim] = (strat_rets.mean() / s * np.sqrt(ann_factor)) if s > 0 else 0.0

    return sharpes


# Run Method 1 against our (simulated) strategy returns
np.random.seed(0)
n_periods = len(strategy_returns)
strategy_sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)

print(f"Strategy Sharpe Ratio:            {strategy_sharpe:.4f}")
print(f"Running Method 1 (5000 sims) ...")

sr_method1 = mc_method1_random_long_short_neutral(
    spy_returns.values, n_periods=n_periods, n_simulations=5000
)
p_method1 = (sr_method1 >= strategy_sharpe).mean()

print(f"\nMethod 1 -- Random positions in {{-1, 0, +1}}:")
print(f"  Random mean Sharpe:   {sr_method1.mean():.4f}")
print(f"  Random 95th pct:      {np.percentile(sr_method1, 95):.4f}")
print(f"  p-value:              {p_method1:.4f}")
print(f"  Reject H0 (5%):       {'Yes' if p_method1 < 0.05 else 'No'}")

### 4.5 Method 2 &mdash; Random Long-Only Positions $\in \{0, +1\}$

$$H_0: \text{Our strategy is no better than any random LONG-ONLY trading strategy}$$

We add a constraint matching our strategy: **long-only**. Positions are drawn from $\{0, +1\}$.

This is a **stricter** test. If the underlying market has a long-term upward drift, *any* strategy with a long bias will profit on average. So we want our strategy to beat random long-only strategies &mdash; not just any random strategy.

In [ ]:
def mc_method2_random_long_only(underlying_returns, n_periods,
                                n_simulations=5000, ann_factor=252):
    """
    Method 2: Random long-only positions in {0, +1} multiplied by reshuffled returns.

    H0: Our strategy is no better than any random long-only trading strategy.
    """
    rets = np.asarray(underlying_returns)
    sharpes = np.zeros(n_simulations)

    for sim in range(n_simulations):
        positions = np.random.choice([0, 1], size=n_periods)
        shuffled_rets = np.random.permutation(rets)[:n_periods]
        strat_rets = positions * shuffled_rets
        s = strat_rets.std()
        sharpes[sim] = (strat_rets.mean() / s * np.sqrt(ann_factor)) if s > 0 else 0.0

    return sharpes


print("Running Method 2 (5000 sims) ...")
sr_method2 = mc_method2_random_long_only(
    spy_returns.values, n_periods=n_periods, n_simulations=5000
)
p_method2 = (sr_method2 >= strategy_sharpe).mean()

print(f"\nMethod 2 -- Random long-only positions in {{0, +1}}:")
print(f"  Random mean Sharpe:   {sr_method2.mean():.4f}")
print(f"  Random 95th pct:      {np.percentile(sr_method2, 95):.4f}")
print(f"  p-value:              {p_method2:.4f}")
print(f"  Reject H0 (5%):       {'Yes' if p_method2 < 0.05 else 'No'}")

### 4.6 Method 3 &mdash; Original Positions $\times$ Reshuffled Returns

$$H_0: \text{Our strategy is no better than a random strategy with same constraints AND same trade characteristics}$$

We **keep our strategy's actual position series** $\text{Pos}^{\text{Orig}}_t$ unchanged (same number of long days, same trade durations, same long bias). We only **reshuffle the underlying returns**:

$$r_t^{\text{Strategy}, H_0} \;=\; \text{Pos}^{\text{Orig}}_t \cdot \tilde{r}_t$$

This is the **strictest** test. It controls for:

- Same proportion of long / short / neutral days
- Same trade frequency
- Same average holding period
- Same long bias

If our strategy still beats this null, it means our **timing** is informative &mdash; not just the average exposure. This is the strongest evidence of skill.

In [ ]:
def mc_method3_keep_positions(positions, underlying_returns,
                              n_simulations=5000, ann_factor=252):
    """
    Method 3: Original strategy positions multiplied by reshuffled underlying returns.

    H0: Our strategy is no better than a random strategy with same
        constraints AND same trade characteristics.
    """
    rets = np.asarray(underlying_returns)
    pos = np.asarray(positions)
    n = min(len(rets), len(pos))
    rets = rets[:n]
    pos = pos[:n]
    sharpes = np.zeros(n_simulations)

    for sim in range(n_simulations):
        shuffled_rets = np.random.permutation(rets)
        strat_rets = pos * shuffled_rets
        s = strat_rets.std()
        sharpes[sim] = (strat_rets.mean() / s * np.sqrt(ann_factor)) if s > 0 else 0.0

    return sharpes


# For Method 3 we need our strategy's actual positions.
# Rebuild SMA(20/100) signal positions on SPY as our running example.
fast_ma = spy_data['Close'].rolling(20).mean()
slow_ma = spy_data['Close'].rolling(100).mean()
positions_orig = (fast_ma > slow_ma).astype(int).shift(1).fillna(0).values
spy_aligned = spy_returns.reindex(spy_data.index).fillna(0).values

# Recompute Sharpe from these actual positions
strategy_rets_aligned = positions_orig * spy_aligned
sr_orig = strategy_rets_aligned.mean() / strategy_rets_aligned.std() * np.sqrt(252)

print(f"SMA(20/100) actual Sharpe:        {sr_orig:.4f}")
print("Running Method 3 (5000 sims) ...")

sr_method3 = mc_method3_keep_positions(
    positions_orig, spy_aligned, n_simulations=5000
)
p_method3 = (sr_method3 >= sr_orig).mean()

print(f"\nMethod 3 -- Same constraints AND same trade characteristics:")
print(f"  Random mean Sharpe:   {sr_method3.mean():.4f}")
print(f"  Random 95th pct:      {np.percentile(sr_method3, 95):.4f}")
print(f"  p-value:              {p_method3:.4f}")
print(f"  Reject H0 (5%):       {'Yes' if p_method3 < 0.05 else 'No'}")

### 4.7 Comparison of the Three Methods

The three p-values together tell a coherent story:

| Method | Null hypothesis | What it controls for | Stringency |
|--------|------------------|-----------------------|------------|
| 1 | Any random strategy           | nothing                                    | weak   |
| 2 | Random long-only strategy     | long bias                                  | medium |
| 3 | Original positions, shuffled returns | trade count, holding period, exposure | strong |

A genuinely skilful strategy should pass Method 3, not just Method 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

methods = [
    (sr_method1, p_method1, strategy_sharpe, "Method 1: Random {-1, 0, +1}"),
    (sr_method2, p_method2, strategy_sharpe, "Method 2: Random Long-Only {0, +1}"),
    (sr_method3, p_method3, sr_orig,         "Method 3: Same Trade Characteristics"),
]

for ax, (sr_arr, pval, ref, title) in zip(axes, methods):
    ax.hist(sr_arr, bins=80, density=True, alpha=0.7, color='grey')
    ax.axvline(ref, color='red', linewidth=2, label=f'Strategy SR={ref:.3f}')
    ax.axvline(np.percentile(sr_arr, 95), color='blue', linestyle='--',
               label=f'95th pct={np.percentile(sr_arr, 95):.3f}')
    ax.set_title(f'{title}\np-value = {pval:.4f}')
    ax.set_xlabel('Sharpe Ratio')
    ax.legend(fontsize=8, loc='upper left')

axes[0].set_ylabel('Density')
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("SKILL vs. LUCK SUMMARY")
print("=" * 60)
print(f"{'Test':<48} {'p-value':>10}")
print("-" * 60)
print(f"{'Method 1: Any random strategy':<48} {p_method1:>10.4f}")
print(f"{'Method 2: Random long-only':<48} {p_method2:>10.4f}")
print(f"{'Method 3: Same trade characteristics':<48} {p_method3:>10.4f}")

## 5. Application to a Backtrader Strategy

Let us apply the full robustness testing framework to an actual backtrader strategy.

In [ ]:
# Run the SMA strategy and collect daily returns
class SMAWithReturns(bt.Strategy):
    params = dict(fast=20, slow=100)
    
    def __init__(self):
        fast_ma = bt.ind.SMA(self.data.close, period=self.p.fast)
        slow_ma = bt.ind.SMA(self.data.close, period=self.p.slow)
        self.crossover = bt.ind.CrossOver(fast_ma, slow_ma)
        self.order = None
        self.daily_returns = []
    
    def notify_order(self, order):
        self.order = None
    
    def next(self):
        # Track daily return
        if len(self) > 1:
            if self.position:
                daily_ret = (self.data.close[0] - self.data.close[-1]) / self.data.close[-1]
            else:
                daily_ret = 0
            self.daily_returns.append(daily_ret)
        
        if self.order:
            return
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.close()

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=spy_data))
cerebro.addstrategy(SMAWithReturns)
cerebro.broker.setcash(100000)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)

results = cerebro.run()
strat = results[0]

bt_returns = pd.Series(strat.daily_returns)
print(f"Strategy generated {len(bt_returns)} daily returns")

# Parametric test
parametric_test(bt_returns, "SMA(20/100) on SPY")

In [ ]:
# Bootstrap
print("\n--- Bootstrap Analysis ---")
boot_sr = simple_bootstrap(bt_returns.values, n_bootstrap=5000, statistic='sharpe')
ci = np.percentile(boot_sr, [2.5, 97.5])
print(f"Bootstrap 95% CI for Sharpe: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"P(SR > 0): {(boot_sr > 0).mean():.1%}")

# Monte Carlo: Methods 2 and 3 (most relevant for a long-only SMA strategy)
actual_sr = bt_returns.mean() / bt_returns.std() * np.sqrt(252)

print("\n--- Monte Carlo: Method 2 (Random Long-Only) ---")
mc_sr_m2 = mc_method2_random_long_only(
    spy_returns.values, n_periods=len(bt_returns), n_simulations=5000
)
p_m2 = (mc_sr_m2 >= actual_sr).mean()
print(f"Strategy SR:        {actual_sr:.4f}")
print(f"Method 2 p-value:   {p_m2:.4f}")

print("\n--- Monte Carlo: Method 3 (Same Trade Characteristics) ---")
fast_ma_app = spy_data['Close'].rolling(20).mean()
slow_ma_app = spy_data['Close'].rolling(100).mean()
sma_positions = (fast_ma_app > slow_ma_app).astype(int).shift(1).fillna(0).values
mc_sr_m3 = mc_method3_keep_positions(
    sma_positions, spy_returns.reindex(spy_data.index).fillna(0).values,
    n_simulations=5000
)
p_m3 = (mc_sr_m3 >= actual_sr).mean()
print(f"Method 3 p-value:   {p_m3:.4f}")

# Final summary
print(f"\n{'='*55}")
print(f"ROBUSTNESS SUMMARY: SMA(20/100) on SPY")
print(f"{'='*55}")
print(f"Parametric p-value:        {stats.ttest_1samp(bt_returns, 0)[1]:.4f}")
print(f"Bootstrap P(SR > 0):       {(boot_sr > 0).mean():.1%}")
print(f"MC Method 2 p-value:       {p_m2:.4f}")
print(f"MC Method 3 p-value:       {p_m3:.4f}")
conclusion = "PASS" if (boot_sr > 0).mean() > 0.9 and p_m3 < 0.10 else "INCONCLUSIVE"
print(f"Overall Assessment:        {conclusion}")

## 6. Exercises

1. **Sampling Distribution Simulation**: Repeat the simulation in Section 2.2 with $Y_i \sim N(\mu = 0.05, \sigma = 0.40)$ and $N \in \{30, 100, 500\}$. Plot the three sampling distributions of the sample mean on the same axes. Verify that the empirical standard deviation of the sample means matches the theoretical $\sigma / \sqrt{N}$.

2. **Parameter Sensitivity**: Choose a strategy from an earlier notebook (e.g., MACD, RSI, Donchian Channel). Create a parameter sensitivity heatmap. What percentage of parameter combinations are profitable?

3. **Bootstrap Confidence Intervals**: Run the SMA(20/100) strategy on three different assets (SPY, EFA, GLD). For each, compute the **simple-block** and **moving-block** bootstrap 95% CI for the Sharpe ratio (block size = 20). Which asset gives the most statistically significant results? Does the choice of block size matter?

4. **Bootstrap for Max Drawdown**: For one of your strategies, compute the bootstrap distribution of the **maximum drawdown** (5%-95% CI). How does the realized drawdown compare to the bootstrap distribution? Is the strategy's drawdown an outlier or typical?

5. **Skill vs. Luck &mdash; All Three Methods**: For one of your strategies, run all three Monte Carlo methods (1, 2, 3) and report the three p-values. Which test does the strategy pass / fail? Interpret what each result implies about the strategy's source of returns.

6. **Skipping Top Trades**: Modify the pairs trading strategy from Notebook 10. Remove the best 5 trades and the best 10 trades. How much does performance change? Is the strategy still profitable?

7. **Fee Sensitivity**: Take any strategy and test it with commissions of 0%, 0.05%, 0.1%, 0.2%, 0.5%. At what commission level does the strategy break even?

8. **Comprehensive Robustness Report**: For one strategy of your choice, create a complete robustness report that includes:
   - Parameter sensitivity heatmap
   - Sub-period analysis (split data into 3 equal periods)
   - Bootstrap (simple, simple-block, moving-block) CIs for return, Sharpe, and max drawdown
   - All three Monte Carlo skill-vs-luck tests
   - Fee sensitivity
   Present your findings in a summary table and decide PASS / INCONCLUSIVE / FAIL.

---
### References
- Harvey, C.R., Liu, Y., & Zhu, H. (2016). *...and the Cross-Section of Expected Returns.* Review of Financial Studies.
- White, H. (2000). *A Reality Check for Data Snooping.* Econometrica.
- Aronson, D. (2007). *Evidence-Based Technical Analysis.* Wiley.
- Efron, B. & Tibshirani, R. (1993). *An Introduction to the Bootstrap.* Chapman & Hall.
- Lo, A. W. (2002). *The Statistics of Sharpe Ratios.* Financial Analysts Journal.
- Bailey, D.H. & López de Prado, M. (2012). *The Sharpe Ratio Efficient Frontier.* Journal of Risk.
- K&uuml;nsch, H. R. (1989). *The Jackknife and the Bootstrap for General Stationary Observations.* Annals of Statistics. *(Moving Block Bootstrap.)*